# Task B-5 & B-6: Model Explainability and Prediction Reasoning

## 1. Model Interpretability (Task B-5)
To understand how Prometheus makes decisions, we analyze which parts of an image contribute most to a detection. For YOLO models, we can visualize the **Confidence Score** and **Feature Maps**.

### Feature Importance
In object detection, "feature importance" relates to the pixels inside and around the bounding box that trigger high activation in the neural network layers.

In [ ]:
import torch
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
import numpy as np

# Load model
model = YOLO('yolov10s.pt')

# Sample frame for explanation
img_path = 'model/data/test_frames/images/train/frame_0000.jpg'
results = model(img_path)

# Visualize detections
res_plotted = results[0].plot()
plt.imshow(cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB))
plt.title("Prometheus Detections")
plt.axis('off')
plt.show()

## 2. Prediction Reasoning (Task B-6)
We provide human-understandable explanations for individual predictions to build trust with gym owners.

### Case Study 1: Person Detection
- **Prediction**: Person (Confidence: 0.92)
- **Reasoning**: The model identified human-like structural patterns (head-shoulder silhouette) in the upper region of the bounding box.

### Case Study 2: Gym Machine Detection
- **Prediction**: Gym-Machine (Confidence: 0.85)
- **Reasoning**: The model detected rigid, metallic structures with specific geometry (cables, weight stacks) characteristic of gym equipment.

### Case Study 3: Low Confidence / Privacy Blur
- **Prediction**: Blurred Region (Privacy Protection)
- **Reasoning**: This area contains a detected face. Following PDPA guidelines, the model applies a Gaussian blur to ensure no individual can be identified from the footage.

In [ ]:
def provide_reasoning(detection_class, confidence):
    reasons = {
        'person': "Detection based on human anatomical features (torso, limbs) and movement patterns.",
        'gym-machine': "Detection based on mechanical geometry, metallic texture, and stationary placement."
    }
    return f"The model is {confidence*100:.1f}% certain this is a {detection_class}. Reasoning: {reasons.get(detection_class, 'General detection.')}"

# Example reasoning for the first 3 detections
for i, box in enumerate(results[0].boxes[:3]):
    cls = model.names[int(box.cls[0])]
    conf = float(box.conf[0])
    print(f"Test Case {i+1}: {provide_reasoning(cls, conf)}")